# MF2 60x Sil 1.3 — Chromatic Z-correction

Measures the axial (Z) and lateral (XY) offset between color channels using
0.1 µm TetraSpek beads imaged through a 0 – 30 µm Z-stack.

**Part 1** — Generates the HAL config and shutter files for the calibration acquisition.  
**Part 2** — Reads the resulting DAX files, localises beads in 3D by Gaussian fitting,
and plots the average displacement of each channel relative to 405 nm.

**Inputs (Part 2):** one or more `.dax` files + matching `.inf` sidecars from the
calibration acquisition, and the `frame_table_*.csv` saved in Part 1.  
**Output:** `zcorrection_beads.csv` with columns
`bead_id, fov_id, 405_x, 405_y, 405_z, 488_x, 488_y, 488_z, 560_x, 560_y, 560_z,
650_x, 650_y, 650_z` (all in µm).

In [ ]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.ndimage  import gaussian_filter, maximum_filter
from scipy.spatial  import KDTree

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/  (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import (
    get_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config,
)
from MERci.acquisition.display  import print_frame_table, display_xml
from MERci.common.io            import parse_inf
from MERci.visualization        import visualize_shutter_sequence

logging.basicConfig(level=logging.WARNING)
print(f"MERCI_DIR  : {MERCI_DIR}")
print(f"SAMPLE_DIR : {SAMPLE_DIR}")

---
## Part 1 — Create HAL config for the calibration acquisition

Generates the shutter and HAL config XML files for a 4-colour, 0 – 30 µm Z-stack
on MF2 using DAX format.  Run this **before** the acquisition.

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

MICROSCOPE    = "MF2"
FILE_TYPE     = ".dax"
EXPOSURE_TIME = 0.3      # seconds

# ── Auto-detect HAL template ───────────────────────────────────────────
_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(
        f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}"
    )
HAL_TEMPLATE = _hal_candidates[0]

# ── Z-stack definition ─────────────────────────────────────────────────
# z_bead is set to -1 µm (outside the data range) so the Z-nanopositioner
# has a reference plane that does not overlap with the bead data stack.
z_bead    = -1.0
bead_seq  = [np.nan]              # one blank (dark) frame as Z reference
color_seq = [405, 488, 560, 650]  # interleaved: all 4 colours per z-plane
end_seq   = [np.nan]              # blank frame to allow Z-nanopositioner return
z_pos     = np.arange(0, 30.2, 0.2)   # 0.0, 0.2, …, 30.0 µm  (151 planes)
SCAN_MODE = "interleaved"

print(f"HAL template : {HAL_TEMPLATE.name}")
print(f"Z-planes     : {len(z_pos)}  ({z_pos[0]:.1f} – {z_pos[-1]:.1f} µm)")
print(f"Total frames : {1 + len(z_pos) * len(color_seq) + 1}  "
      f"(1 bead + {len(z_pos)}×{len(color_seq)} data + 1 end)")

In [ ]:
frame_table = get_frame_table(
    z_bead, bead_seq, color_seq, end_seq, z_pos,
    microscope=MICROSCOPE, scan_mode=SCAN_MODE,
)
name = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)
print(f"Config name : {name}")
print_frame_table(frame_table)

# ── Save frame table ───────────────────────────────────────────────────
ft_path = METADATA_DIR / f"frame_table_{name}.csv"
frame_table.to_csv(ft_path)
print(f"Frame table : {ft_path}")

# ── Write shutter file ─────────────────────────────────────────────────
shutter_name = f"shutter-{name}.xml"
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path)
print(f"Shutter     : {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config ───────────────────────────────────────────────────
hal_name   = f"hal-config-{MICROSCOPE.lower()}-zcal-{name}.xml"
hal_output = SETTINGS_DIR / hal_name
create_hal_config(
    HAL_TEMPLATE, frame_table, shutter_name, hal_output,
    file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME,
)
print(f"HAL config  : {hal_output}")
display_xml(hal_output)

In [ ]:
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

---
## Part 2 — Bead localisation and chromatic shift analysis

Reads a list of DAX files, detects TetraSpek beads in each colour channel via
3D Gaussian fitting, and measures the average XYZ displacement of 488, 560, and
650 nm relative to the 405 nm reference channel.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
# Edit DAX_FILES to point to the files acquired with the config above.
DAX_FILES = [
    SAMPLE_DIR / "data" / "zcal_fov001.dax",
    # add more files as needed
]

# Frame table produced in Part 1 (auto-detected from metadata/ if name is defined)
try:
    FRAME_TABLE_CSV = ft_path       # if Part 1 was run in the same session
except NameError:
    FRAME_TABLE_CSV = sorted(METADATA_DIR.glob("frame_table_405f*-488f*.csv"))[-1]

OUTPUT_CSV = SAMPLE_DIR / "metadata" / "zcorrection_beads.csv"

# ── Acquisition constants ──────────────────────────────────────────────
PIXEL_SIZE_UM = 0.109   # µm / pixel  (MF2 with 60x silicone 1.3 NA)
Z_STEP_UM     = 0.2     # µm per z-plane
REF_COLOR     = 405     # reference channel for shift measurement

# ── Bead detection ─────────────────────────────────────────────────────
DETECT_MIN_DIST_PX  = 15    # minimum centre-to-centre separation (pixels)
DETECT_THRESH_SIGMA = 3.0   # detection threshold above background (sigma)

# ── 3D Gaussian crop ───────────────────────────────────────────────────
CROP_XY_PX = 10   # half-width of XY crop around each bead centre (pixels)
CROP_Z_PL  = 12   # half-width of Z crop around the rough z-peak (planes, 2.4 µm)

# ── Bead matching ──────────────────────────────────────────────────────
MATCH_TOL_PX = 5.0  # maximum XY distance (pixels) to match a bead across colours

print(f"Frame table : {FRAME_TABLE_CSV.name}")
print(f"DAX files   : {len(DAX_FILES)}")

In [ ]:
# ── DAX I/O helpers ────────────────────────────────────────────────────

def read_dax_crop(dax_path, frame_indices, H, W, r1, r2, c1, c2):
    """
    Read a spatial crop [r1:r2, c1:c2] from each frame in *frame_indices*.

    Returns (n_z, crop_h, crop_w) uint16 array.  Only the required rows are
    read from disk — much more memory-efficient than loading full frames.
    """
    n_z       = len(frame_indices)
    crop_h    = r2 - r1
    crop_w    = c2 - c1
    out       = np.zeros((n_z, crop_h, crop_w), dtype=np.uint16)
    row_bytes = W * 2                 # bytes per full row (uint16)
    frm_bytes = H * W * 2             # bytes per full frame
    with open(dax_path, "rb") as fh:
        for zi, idx in enumerate(frame_indices):
            frm_offset = int(idx) * frm_bytes
            for ri, row in enumerate(range(r1, r2)):
                fh.seek(frm_offset + row * row_bytes + c1 * 2)
                out[zi, ri] = np.frombuffer(fh.read(crop_w * 2), dtype=np.uint16)
    return out


def compute_max_projection(dax_path, frame_indices, H, W):
    """
    Stream frames from *dax_path* one at a time and return the
    z-maximum projection as a (H, W) float32 array.
    """
    proj      = np.zeros((H, W), dtype=np.float32)
    frm_bytes = H * W * 2
    with open(dax_path, "rb") as fh:
        for idx in frame_indices:
            fh.seek(int(idx) * frm_bytes)
            frame = np.frombuffer(fh.read(frm_bytes), dtype=np.uint16
                                  ).reshape(H, W).astype(np.float32)
            np.maximum(proj, frame, out=proj)
    return proj

In [ ]:
# ── 3D Gaussian fitting ────────────────────────────────────────────────

def _gauss3d(coords, amp, x0, y0, z0, sig_xy, sig_z, offset):
    """Evaluate a 3D Gaussian on a flattened coordinate array."""
    x, y, z = coords
    return (
        amp * np.exp(
            -((x - x0)**2 + (y - y0)**2) / (2 * sig_xy**2)
            - (z - z0)**2              / (2 * sig_z **2)
        ) + offset
    ).ravel()


def fit_bead_3d(sub_vol, z_um_vals):
    """
    Fit a 3D Gaussian to *sub_vol* and return the fitted (x_px, y_px, z_um).

    Parameters
    ----------
    sub_vol   : (n_z, n_y, n_x) float32 array — cropped intensity volume
    z_um_vals : (n_z,) float array — z positions in µm for each plane

    Returns
    -------
    (x_px, y_px, z_um) relative to the crop origin, or None if fit fails.
    """
    n_z, n_y, n_x = sub_vol.shape
    if n_z < 3 or n_y < 3 or n_x < 3:
        return None

    # Coordinate grids: x and y in pixels, z in µm
    x_arr = np.arange(n_x, dtype=float)
    y_arr = np.arange(n_y, dtype=float)
    z_arr = z_um_vals.astype(float)

    # meshgrid with indexing matching (z, y, x) array layout
    zg, yg, xg = np.meshgrid(z_arr, y_arr, x_arr, indexing="ij")  # each (n_z, n_y, n_x)

    z_peak = int(np.unravel_index(sub_vol.argmax(), sub_vol.shape)[0])
    amp    = float(sub_vol.max() - sub_vol.min())
    p0     = [amp, n_x / 2.0, n_y / 2.0, float(z_arr[z_peak]),
               1.5, 0.6, float(sub_vol.min())]
    lo     = [0,   0,         0,         z_arr[0],  0.1, 0.1, -np.inf]
    hi     = [np.inf, n_x,    n_y,       z_arr[-1], 8.0, 5.0,  np.inf]

    try:
        popt, _ = curve_fit(
            _gauss3d,
            (xg.ravel(), yg.ravel(), zg.ravel()),
            sub_vol.ravel().astype(float),
            p0=p0, bounds=(lo, hi), maxfev=20_000,
        )
        return float(popt[1]), float(popt[2]), float(popt[3])   # x_px, y_px, z_um
    except Exception:
        return None

In [ ]:
# ── Bead detection and localisation ────────────────────────────────────

def detect_beads_2d(max_proj, min_dist_px, thresh_sigma):
    """
    Find bead centres in a z-max projection.

    Returns (N, 2) integer array of (row, col) positions.
    """
    blurred  = gaussian_filter(max_proj.astype(float), sigma=1.5)
    bg_mask  = blurred < np.percentile(blurred, 80)
    bg_med   = np.median(blurred[bg_mask])
    bg_std   = blurred[bg_mask].std()
    thresh   = bg_med + thresh_sigma * bg_std
    local_mx = maximum_filter(blurred, size=int(min_dist_px)) == blurred
    return np.argwhere(local_mx & (blurred > thresh))


def localize_beads_in_file(dax_path, frame_table, crop_xy, crop_z,
                            min_dist_px, thresh_sigma):
    """
    Detect and 3D-Gaussian-fit all beads in every colour channel of one DAX file.

    Returns
    -------
    dict  {color_nm: pd.DataFrame(columns=['x_px', 'y_px', 'z_um'])}
    """
    inf = parse_inf(dax_path)
    H, W = int(inf["frame_height"]), int(inf["frame_width"])
    colors = sorted(
        c for c in frame_table["color"].dropna().unique()
        if not np.isnan(float(c))
    )
    results = {}

    for color in colors:
        color_int  = int(color)
        mask       = frame_table["color"] == color
        f_indices  = frame_table.index[mask].tolist()
        z_um_vals  = frame_table.loc[mask, "z"].values.astype(float)

        # ── Step 1: z-max projection for 2-D bead detection ──────────
        max_proj   = compute_max_projection(dax_path, f_indices, H, W)
        candidates = detect_beads_2d(max_proj, min_dist_px, thresh_sigma)
        print(f"  {color_int} nm : {len(candidates)} candidates")

        # ── Step 2: 3-D Gaussian fit for each candidate ───────────────
        rows = []
        for (r0, c0) in candidates:
            r1 = max(0,  r0 - crop_xy);  r2 = min(H, r0 + crop_xy)
            c1 = max(0,  c0 - crop_xy);  c2 = min(W, c0 + crop_xy)

            # Load the full Z-stack for this small XY crop
            vol = read_dax_crop(dax_path, f_indices, H, W, r1, r2, c1, c2
                                ).astype(np.float32)

            # Rough z-peak from the mean z-profile of the crop
            z_profile  = vol.mean(axis=(1, 2))
            z_peak_idx = int(np.argmax(z_profile))
            z1 = max(0, z_peak_idx - crop_z)
            z2 = min(len(f_indices), z_peak_idx + crop_z)

            fit = fit_bead_3d(vol[z1:z2], z_um_vals[z1:z2])
            if fit is None:
                continue
            lx, ly, z_um = fit
            rows.append({"x_px": c1 + lx, "y_px": r1 + ly, "z_um": z_um})

        results[color_int] = pd.DataFrame(rows)
        print(f"         {len(rows)} beads fitted")

    return results


def match_beads_across_colors(color_dfs, ref_color, match_tol_px, pixel_size_um):
    """
    Match beads across colour channels by XY proximity.

    Uses *ref_color* as the reference; keeps only beads found in all channels.
    Positions in the output are in µm (x, y from pixel coordinates; z already µm).

    Returns pd.DataFrame with columns
    bead_id, {ref}_x, {ref}_y, {ref}_z, {c2}_x, … for every colour.
    """
    ref_df     = color_dfs[ref_color].reset_index(drop=True)
    ref_tree   = KDTree(ref_df[["x_px", "y_px"]].values)
    all_colors = sorted(color_dfs.keys())

    # Map: ref_index → per-color row
    matched = {ref_color: {i: ref_df.loc[i] for i in ref_df.index}}
    for color in all_colors:
        if color == ref_color:
            continue
        q_df   = color_dfs[color]
        if q_df.empty:
            matched[color] = {}
            continue
        dists, idxs = ref_tree.query(q_df[["x_px", "y_px"]].values, k=1)
        matched[color] = {}
        for qi, (dist, ref_i) in enumerate(zip(dists, idxs)):
            if dist < match_tol_px:
                matched[color][ref_i] = q_df.iloc[qi]

    # Keep only beads found in every channel
    valid_idx = [
        i for i in ref_df.index
        if all(i in matched[c] for c in all_colors)
    ]

    rows = []
    for bead_id, ref_i in enumerate(valid_idx):
        row = {"bead_id": bead_id}
        for color in all_colors:
            r = matched[color][ref_i]
            row[f"{color}_x"] = float(r["x_px"]) * pixel_size_um
            row[f"{color}_y"] = float(r["y_px"]) * pixel_size_um
            row[f"{color}_z"] = float(r["z_um"])
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# ── Main analysis loop ─────────────────────────────────────────────────

frame_table = pd.read_csv(FRAME_TABLE_CSV, index_col=0)

all_results = []
for fov_id, dax_path in enumerate(DAX_FILES):
    print(f"\nFOV {fov_id}  —  {dax_path.name}")

    color_dfs = localize_beads_in_file(
        dax_path, frame_table,
        crop_xy=CROP_XY_PX, crop_z=CROP_Z_PL,
        min_dist_px=DETECT_MIN_DIST_PX, thresh_sigma=DETECT_THRESH_SIGMA,
    )

    df_fov = match_beads_across_colors(
        color_dfs, ref_color=REF_COLOR,
        match_tol_px=MATCH_TOL_PX, pixel_size_um=PIXEL_SIZE_UM,
    )
    df_fov.insert(1, "fov_id", fov_id)
    all_results.append(df_fov)
    print(f"  {len(df_fov)} beads matched across all channels")

beads_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# Re-number bead_id globally
beads_df["bead_id"] = np.arange(len(beads_df))

beads_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nTotal beads : {len(beads_df)}")
print(f"Saved       : {OUTPUT_CSV}")
beads_df.head()

---
## Part 3 — Chromatic shift visualisation

Average displacement of each channel relative to 405 nm (reference).
Error bars show the standard deviation across all beads.

In [ ]:
# ── Bar charts: mean shift per channel ────────────────────────────────

compare_colors = [c for c in [488, 560, 650] if f"{c}_x" in beads_df.columns]

shift_data = []
for color in compare_colors:
    dx = beads_df[f"{color}_x"] - beads_df[f"{REF_COLOR}_x"]
    dy = beads_df[f"{color}_y"] - beads_df[f"{REF_COLOR}_y"]
    dz = beads_df[f"{color}_z"] - beads_df[f"{REF_COLOR}_z"]
    shift_data.append({
        "color": color,
        "dx_mean": dx.mean(), "dx_std": dx.std(),
        "dy_mean": dy.mean(), "dy_std": dy.std(),
        "dz_mean": dz.mean(), "dz_std": dz.std(),
    })

shift_df = pd.DataFrame(shift_data)
print(shift_df.to_string(index=False))

_COLOR_HEX = {488: "#1f77b4", 560: "#ff7f0e", 650: "#2ca02c"}
x_pos = np.arange(len(compare_colors))
xlabels = [f"{c} nm" for c in compare_colors]

fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=False)
for ax, axis_label, mean_col, std_col in zip(
    axes,
    ["\u0394X (µm)", "\u0394Y (µm)", "\u0394Z (µm)"],
    ["dx_mean", "dy_mean", "dz_mean"],
    ["dx_std",  "dy_std",  "dz_std"],
):
    colors_hex = [_COLOR_HEX.get(c, "#7f7f7f") for c in compare_colors]
    ax.bar(x_pos, shift_df[mean_col], yerr=shift_df[std_col],
           color=colors_hex, edgecolor="k", linewidth=0.8,
           capsize=5, error_kw={"linewidth": 1.2})
    ax.axhline(0, color="k", linewidth=0.8, linestyle="--")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(xlabels)
    ax.set_ylabel(axis_label)
    ax.set_title(f"Mean {axis_label} vs {REF_COLOR} nm")

fig.suptitle(
    f"Chromatic shift  |  {len(beads_df)} beads, {len(DAX_FILES)} FOV(s)",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
fig.savefig(
    SAMPLE_DIR / "metadata" / "zcorrection_shifts.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

In [ ]:
# ── Optional: ΔZ vs z_405 — detects field-dependent z-curvature ───────

fig, axes = plt.subplots(1, len(compare_colors),
                         figsize=(5 * len(compare_colors), 4), sharey=False)
if len(compare_colors) == 1:
    axes = [axes]

for ax, color in zip(axes, compare_colors):
    z_ref = beads_df[f"{REF_COLOR}_z"]
    dz    = beads_df[f"{color}_z"] - z_ref
    ax.scatter(z_ref, dz, s=12, alpha=0.5, color=_COLOR_HEX.get(color, "#7f7f7f"))
    # linear trend
    if len(z_ref) > 1:
        coeff = np.polyfit(z_ref, dz, 1)
        z_line = np.linspace(z_ref.min(), z_ref.max(), 100)
        ax.plot(z_line, np.polyval(coeff, z_line), "k--", linewidth=1.2)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    ax.set_xlabel(f"z_405 (µm)")
    ax.set_ylabel(f"\u0394Z  {color} \u2013 405 (µm)")
    ax.set_title(f"{color} nm  vs  {REF_COLOR} nm")

fig.suptitle("Z-shift vs reference z position", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(
    SAMPLE_DIR / "metadata" / "zcorrection_zdependent.png",
    dpi=200, bbox_inches="tight",
)
plt.show()